In [1]:
from llm_model.call_model import ModelCaller

In [2]:
#model_name = "openai/gpt-4.1"
model_name="gpt-4o-mini"
llm = ModelCaller(model_name , max_tokens=500)
#print(llm.call_model("Tell me about Deep space?"))

/Users/suryaatul/PythonWorkspace/AgenticAI/Notebooks/llm_model/call_model.py:23: ExperimentalWarning: AzureAIChatCompletionsModel is currently in preview and is subject to change. This preview is provided without a service-level agreement, and we don't recommend it for production workloads. Certain features might not be supported or might have constrained capabilities. For more information, see https://azure.microsoft.com/support/legal/preview-supplemental-terms
  model = AzureAIChatCompletionsModel(


### ----- Addition Tool ------------

In [3]:
import re
from langchain_core.tools import StructuredTool , tool

def sum_numbers_from_text2(inputs: str , average :bool , absolute:bool) -> float:
    """
    Adds a list of numbers provided in the input string , and calcuate average if flag is set to true.
    
    Args:
        text: A string containing numbers that should be extracted and summed.
        boolean : A boolean value in True or False to calculate the average of extracted and summed value
        boolean : boolean flag to incidicate of the value to return should be absolute
        
    Returns:
        The sum of all numbers found in the input if boolean flag is false
        the average of the numbers in the input if boolean flag is true
    """
    # Use regular expressions to extract all numbers from the input
    numbers = [int(num) for num in re.findall(r'\d+', inputs)]
  
    try:

        if not numbers:
            raise ValueError("No numbers in the input string.")
        else:
            result = sum(numbers)

        if absolute:
            result = abs(result)
        
        if average == False:
            return result
        else:
            return result/len(numbers)
    except Exception as e:
        print(f"Error caught as {e}")

In [4]:
from pydantic import BaseModel,Field


class SumInput(BaseModel):
    inputs: str=Field(description="Input String")
    average: bool = Field(description = " boolean value to calculate average of numbers passed")
    absolute: bool = Field(description = " boolean flag to indicate if value should be absolute")

In [5]:
from langchain_core.tools import StructuredTool

add_tool = StructuredTool.from_function(
    func=sum_numbers_from_text2,
    name="addition_tool", # Define name explicitly
    description="Useful for identifying numeric values from string and sum them ", # Define description
    args_schema=SumInput
)

### --- Substraction Tool --------------

In [31]:

def subtract_numbers(inputs: str) -> dict:
    """
    Extracts numbers from a string, negates the first number, and successively subtracts 
    the remaining numbers in the list.

    This function is designed to handle input in string format, where numbers are separated 
    by spaces, commas, or other delimiters. It parses the string, extracts valid numeric values, 
    and performs a step-by-step subtraction operation starting with the first number negated.

    Parameters:
    - inputs (str): 
      A string containing numbers to subtract. The string may include spaces, commas, or 
      other delimiters between the numbers.

    Returns:
    - dict: 
      A dictionary containing the key "result" with the calculated difference as its value. 
      If no valid numbers are found in the input string, the result defaults to 0.

    Example Input:
    "100, 20, 10"

    Example Output:
    {"result": -130}

    Notes:
    - Non-numeric characters in the input are ignored.
    - If the input string contains only one valid number, the result will be that number negated.
    - Handles a variety of delimiters (e.g., spaces, commas) but does not validate input formats 
      beyond extracting numeric values.
    """
    # Extract numbers from the string
    numbers = [int(num) for num in inputs.replace(",", "").split() if num.isdigit()]

    # If no numbers are found, return 0
    if not numbers:
        return {"result": 0}

    # Start with the first number negated
    result = numbers[0]

    # Subtract all subsequent numbers
    for num in numbers[1:]:
        result -= num

    return {"result": result}

In [32]:
from pydantic import BaseModel,Field
from langchain_core.tools import StructuredTool

class SubInput(BaseModel):
    inputs: str=Field(description="Input String")


sub_tool = StructuredTool.from_function(
    func=subtract_numbers,
    name="substraction_tool", # Define name explicitly
    description="Useful for identifying numeric values from string and substract them ", # Define description
    args_schema=SubInput
)

In [33]:
print("Name: \n", sub_tool.name)
print("Description: \n", sub_tool.description) 
print("Args: \n", sub_tool.args) 

Name: 
 substraction_tool
Description: 
 Useful for identifying numeric values from string and substract them
Args: 
 {'inputs': {'description': 'Input String', 'title': 'Inputs', 'type': 'string'}}


In [34]:
print("Calling Tool Function:")
test_input = "10 20 30 and four a b" 
print(sub_tool.invoke(test_input))  # Example

Calling Tool Function:
{'result': -40}


### --- Multiply Tool --------

In [14]:
from langchain_core.tools import   tool
import re

# Multiplication Tool
@tool
def multiply_numbers(inputs: str) -> dict:
    """
    Extracts numbers from a string and calculates their product.

    Parameters:
    - inputs (str): A string containing numbers separated by spaces, commas, or other delimiters.

    Returns:
    - dict: A dictionary with the key "result" containing the product of the numbers.

    Example Input:
    "2, 3, 4"

    Example Output:
    {"result": 24}

    Notes:
    - If no numbers are found, the result defaults to 1 (neutral element for multiplication).
    """
    # Extract numbers from the string
    numbers = [int(num) for num in inputs.replace(",", "").split() if num.isdigit()]
    print(numbers)

    # If no numbers are found, return 1
    if not numbers:
        return {"result": 1}

    # Calculate the product of the numbers
    result = 1
    for num in numbers:
        result *= num
        print(num)

    return {"result": result}

### --- Multiply Tool --------

In [19]:
from langchain_core.tools import   tool
import re

# Division Tool
@tool
def divide_numbers(inputs: str) -> dict:
    """
    Extracts numbers from a string and calculates the result of dividing the first number 
    by the subsequent numbers in sequence.

    Parameters:
    - inputs (str): A string containing numbers separated by spaces, commas, or other delimiters.

    Returns:
    - dict: A dictionary with the key "result" containing the quotient.

    Example Input:
    "100, 5, 2"

    Example Output:
    {"result": 10.0}

    Notes:
    - If no numbers are found, the result defaults to 0.
    - Division by zero will raise an error.
    """
    # Extract numbers from the string
    numbers = [int(num) for num in inputs.replace(",", "").split() if num.isdigit()]


    # If no numbers are found, return 0
    if not numbers:
        return {"result": 0}

    # Calculate the result of dividing the first number by subsequent numbers
    result = numbers[0]
    for num in numbers[1:]:
        result /= num

    return {"result": result}

In [20]:
# Testing multiply_tool
multiply_test_input = "2, 3, and four "
multiply_result = multiply_numbers.invoke(multiply_test_input)
print("--- Testing MultiplyTool ---")
print(f"Input: {multiply_test_input}")
print(f"Output: {multiply_result}")

[2, 3]
2
3
--- Testing MultiplyTool ---
Input: 2, 3, and four 
Output: {'result': 6}


In [21]:
# Testing multiply_tool
divide_numbers_inputs = "2, 3, and four "
divide_numbers_output = divide_numbers.invoke(multiply_test_input)
print("--- Testing MultiplyTool ---")
print(f"Input: {divide_numbers_inputs}")
print(f"Output: {divide_numbers_output}")

--- Testing MultiplyTool ---
Input: 2, 3, and four 
Output: {'result': 0.6666666666666666}


## Building the agent

With the implementation of mathematical operators—addition, subtraction, multiplication, and division — you have established a simple yet functional mathematical toolkit. Unlike before, the agent must now not only select the appropriate tool and process the input but also determine the correct mathematical operation based on the user's query.

Let's create the agent object. first, combine all tools into a single list:

In [35]:
tools = [multiply_numbers , divide_numbers , sub_tool , add_tool]

In [36]:
from langchain.agents import create_agent 

math_agent = create_agent(model=llm.get_model(),
                         tools=tools)



In [27]:
from langchain_core.messages import HumanMessage , AIMessage

response = math_agent.invoke({
    "messages": [
        AIMessage(content="you are wonderful assistant to perform calculations for numbers in input string"),
        HumanMessage(content="Add the number -10, -20, -30 and calculate average")
    ]
})

# Get the final answer
final_answer = response["messages"][-1].content
print(final_answer)

The average of the numbers -10, -20, and -30 is 20.0.


In [28]:
from langchain_core.messages import HumanMessage , AIMessage

response = math_agent.invoke({
    "messages": [
        AIMessage(content="you are wonderful assistant to perform the calculations for numbers in input string"),
        HumanMessage(content="What is 25 divided by 4?")
    ]
})

# Get the final answer
final_answer = response["messages"][-1].content
print(final_answer)

25 divided by 4 is 6.25.


In [37]:
response_2 = math_agent.invoke({
    "messages": [
        AIMessage(content="you are wonderful assistant to perform the calculations for numbers in input string"),
        HumanMessage(content="Subtract 100, 20, and 10.")
    ]
})

# Get the final answer
final_answer_2 = response_2["messages"][-2].content
print(final_answer_2)

{"result": 70}


In [38]:
print("\n--- Testing MultiplyTool ---")
response = math_agent.invoke({
    "messages": [
        ("human", "Multiply 2, 3, and four.")
    ]
})
print("Agent Response:", response["messages"][-1].content)

print("\n--- Testing DivideTool ---")
response = math_agent.invoke({
    "messages": [
        ("human", "Divide 100 by 5 and then by 2.")
    ]
})
print("Agent Response:", response["messages"][-1].content)


--- Testing MultiplyTool ---
[2, 3, 4]
2
3
4
Agent Response: The product of 2, 3, and 4 is 24.

--- Testing DivideTool ---
Agent Response: The result of dividing 100 by 5 and then by 2 is 10.0.


In [70]:
print(response['messages'][-2].content)

{"result": 10.0}


In [ ]:
print(response['messages'][1].tool_calls[-1]['name'])

In [72]:
# Test Cases
test_cases = [
    {
        "query": "Subtract 100, 20, and 10.",
        "expected": {"result": 70},
        "description": "Testing subtraction tool with sequential subtraction."
    },
    {
        "query": "Multiply 2, 3, and 4.",
        "expected": {"result": 24},
        "description": "Testing multiplication tool for a list of numbers."
    },
    {
        "query": "Divide 100 by 5 and then by 2.",
        "expected": {"result": 10.0},
        "description": "Testing division tool with sequential division."
    },
    {
        "query": "Subtract 50 from 20.",
        "expected": {"result": -30},
        "description": "Testing subtraction tool with negative results."
    }
]



In [92]:
from langchain_core.messages import HumanMessage , AIMessage

correct_tasks = []
# Corrected test execution
for index, test in enumerate(test_cases, start=1):
    query = test["query"]
    expected_result = test["expected"]["result"]  # Extract just the value
    
    print(f"\n--- Test Case {index}: {test['description']} ---")
    print(f"Query: {query}")
    
    # Properly format the input
    response = math_agent.invoke({"messages": 
                                  [
                                    AIMessage(content="you are wonderful assistant to perform the calculations for numbers in input string"),
                                    HumanMessage(content=query)
                                    ]
                                 })
    
    # Find the tool message in the response
    tool_message = None
    for msg in response["messages"]:
        if hasattr(msg, 'name') and msg.name in ['multiply_numbers' , 'divide_numbers' , 'subtract_numbers' , 'add_tool']:
            tool_message = msg
            break
    
    if tool_message:
        # Parse the tool result from its content
        import json
        tool_result = json.loads(tool_message.content)["result"]
        print(f"Tool Result: {tool_result}")
        print(f"Expected Result: {expected_result}")
        
        if tool_result == expected_result:
            print(f"✅ Test Passed: {test['description']}")
            correct_tasks.append(test["description"])
        else:
            print(f"❌ Test Failed: {test['description']}")
    else:
        print("❌ No tool was called by the agent")

print("\nCorrectly passed tests:", correct_tasks)


--- Test Case 1: Testing subtraction tool with sequential subtraction. ---
Query: Subtract 100, 20, and 10.


KeyboardInterrupt: 

In [91]:
for msg in response["messages"]:
    print(msg)
    print("\n")
    

content='Subtract 100, 20, and 10.' additional_kwargs={} response_metadata={} id='9640a7b9-09de-4bf7-b2d3-91d56faac899'


content='' additional_kwargs={} response_metadata={'model': 'gpt-4o-mini-2024-07-18', 'token_usage': {'input_tokens': 357, 'output_tokens': 22, 'total_tokens': 379}, 'finish_reason': 'tool_calls'} id='lc_run--019c1448-9c39-7172-9504-f8a014d01ed6-0' tool_calls=[{'name': 'substraction_tool', 'args': {'inputs': '100, 20, 10'}, 'id': 'call_T0GgrJ06WnUARxPYtG75S3m1', 'type': 'tool_call'}] invalid_tool_calls=[] usage_metadata={'input_tokens': 357, 'output_tokens': 22, 'total_tokens': 379}


content='{"result": 70}' name='substraction_tool' id='0af20229-783d-4bef-bb67-5aa2cb505993' tool_call_id='call_T0GgrJ06WnUARxPYtG75S3m1'


content='The result of subtracting 20 and 10 from 100 is 70.' additional_kwargs={} response_metadata={'model': 'gpt-4o-mini-2024-07-18', 'token_usage': {'input_tokens': 393, 'output_tokens': 19, 'total_tokens': 412}, 'finish_reason': 'stop'} id='lc_

In [87]:
print(response["messages"][1])

content='' additional_kwargs={} response_metadata={'model': 'gpt-4o-mini-2024-07-18', 'token_usage': {'input_tokens': 357, 'output_tokens': 22, 'total_tokens': 379}, 'finish_reason': 'tool_calls'} id='lc_run--019c1448-9c39-7172-9504-f8a014d01ed6-0' tool_calls=[{'name': 'substraction_tool', 'args': {'inputs': '100, 20, 10'}, 'id': 'call_T0GgrJ06WnUARxPYtG75S3m1', 'type': 'tool_call'}] invalid_tool_calls=[] usage_metadata={'input_tokens': 357, 'output_tokens': 22, 'total_tokens': 379}
